In [1]:
# ============================================================
# KAGGLE ONE-CELL
# KS MULTI-FIDELITY QUERY-t ICL TRAINING -- LARGE MODEL 288-H6-FF576
# FD2-LF72 / SPECTRAL-HF256 / WITH-t / MIXED-M / PAD23
# ============================================================
#
# DATASET
# -------
# PDE:
#   u_t + u u_x + u_xx + beta u_xxxx = 0,
#   x in [0, 2*pi), periodic.
#
# Formal dataset:
#   beta: 200 uniformly spaced operators in [0.070, 0.085]
#   trajectories/operator: 20
#   train/test operators: 140 / 60 (use exact split saved in NPZ)
#   LF: N=72, second-order centered FD + ETDRK4
#   HF: N=256, 3/2-dealiased Fourier pseudospectral + ETDRK4
#   common learning grid: Nr=100
#   saved times: 0.00, 0.05, ..., 3.50 (71 snapshots)
#   eligible demo/query pool: 2.50, 2.55, ..., 3.50 (21 time slots)
#
# ICL TASK
# --------
# For one fixed (beta,u0) trajectory:
#
#   M temporal demonstrations:
#       [x, t_d1, uc(t_d1), uf(t_d1),
#        ...
#        x, t_dM, uc(t_dM), uf(t_dM)]
#
#   query:
#       [x, t_q, uc(t_q)]
#
#   target:
#       uf(t_q)
#
# All M demonstrations and the query come from the SAME trajectory.
# M ~ Uniform{1,2,3,4,5} independently for every training episode.
# The M+1 time slots are distinct and remain in random sampled order.
#
# Sequence length:
#   L_M = 4M + 3
# so M=5 gives 23 rows.
#
# Mixed-M batches are right-padded to 23 rows and use a boolean
# src_key_padding_mask. Prediction is gathered from each episode's
# final valid query-uc row, i.e. zero-based row 4M+2.
#
# TRAINING
# --------
#   batch size       = 1000
#   steps / epoch    = 900
#   epochs           = 150
#   AdamW wd         = 1e-4
#   LR               = 1e-4 -> 2e-4 warmup -> cosine -> 1e-5
#   warmup fraction  = 0.08
#   grad clip        = 1.0
#   FP32
#
# MODEL
# -----
#   d_model = 288
#   heads   = 6
#   layers  = 5
#   FF      = 576
#   dropout = 0.10
#   expected trainable parameters = 3,484,036
#
# EVALUATION
# ----------
#   - fixed 512-case M=5 TRAIN monitor selects BEST checkpoint
#   - fixed unseen-operator diagnostic:
#         60 test operators x 16 cases/operator = 960 cases
#   - independent formal test after training:
#         1024 unseen-operator M=5 cases on BEST checkpoint
#
# DATASET ROOT
# ------------
#   /kaggle/input/notebooks/xinyuanli913/ks-eqn-dataset
#
# Exact expected NPZ:
#   ks_mf_queryt_beta070_085_FD2LF72_SpecHF256_T3p5_
#   Nop200_Ntraj20_Nt71_Nr100.npz
# ============================================================

from __future__ import annotations

import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn


# ============================================================
# 0. CONFIG
# ============================================================

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATASET_ROOT = Path(
    "/kaggle/input/notebooks/xinyuanli913/ks-eqn-dataset"
)

EXPECTED_DATASET_NAME = (
    "ks_mf_queryt_beta070_085_"
    "FD2LF72_SpecHF256_T3p5_"
    "Nop200_Ntraj20_Nt71_Nr100.npz"
)

RUN_TAG = (
    "ks_queryt_withT_beta070_085_T3p5_"
    "FD2LF72_SpecHF256_Nr100_"
    "large288_H6_L5_FF576_train140_test60_"
    "sameTraj_randomTime_mixedM_pad23_150ep_step900"
)

SAVE_DIR = Path("/kaggle/working") / RUN_TAG
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Exact operator split is read from the NPZ.
N_TRAIN_OP = 140
N_TEST_OP = 60

# ICL
MIN_DEMOS = 1
MAX_DEMOS = 5
TEST_FIXED_DEMOS = 5

# Training
BATCH_SIZE = 1000
EPOCHS = 150
STEPS_PER_EPOCH = 900
PRINT_EVERY = 5

# Fixed training-side monitor
N_MONITOR_CASES = 512

# Fixed unseen-beta diagnostic
TEST_CASES_PER_OPERATOR = 16
TEST_EVERY = 5
TEST_CASE_SEED = SEED + 301

EVAL_BATCH_SIZE = 256

# Independent formal heldout evaluation after training
FORMAL_TEST_CASES = 1024
FORMAL_TEST_CASE_SEED = SEED + 1301

# Optimizer / scheduler
WEIGHT_DECAY = 1e-4
BASE_LR = 1e-4
MAX_LR = 2e-4
MIN_LR = 1e-5
WARMUP_FRAC = 0.08
GRAD_CLIP = 1.0

# Model
D_MODEL = 288
N_HEAD = 6
N_LAYERS = 5
D_FF = 576
DROPOUT = 0.10
EXPECTED_PARAMS = 3_484_036

EPS_METRIC = 1e-8

# Expected dataset facts
N_OP_EXPECTED = 200
NC_NATIVE_EXPECTED = 72
NF_NATIVE_EXPECTED = 256
NR_SHARED_EXPECTED = 100
T_EXPECTED = 3.5
N_TRAJ_EXPECTED = 20
N_TIME_EXPECTED = 71
SNAPSHOT_DT_EXPECTED = 0.05
N_ELIGIBLE_TIME_EXPECTED = 21
MIN_ELIGIBLE_T_EXPECTED = 2.50
MAX_ELIGIBLE_T_EXPECTED = 3.50

LF_SOLVER_TAG_EXPECTED = "KS_FD2_CENTERED_ETDRK4_N72_V1"
HF_SOLVER_TAG_EXPECTED = "KS_SPECTRAL_3OVER2_ETDRK4_N256_V1"

# Sequence row types
TYPE_X_DEMO = 0
TYPE_T_DEMO = 1
TYPE_UC_DEMO = 2
TYPE_UF_DEMO = 3
TYPE_X_QUERY = 4
TYPE_T_QUERY = 5
TYPE_UC_QUERY = 6
N_ROW_TYPES = 7

MAX_CTX_ROWS = 4 * MAX_DEMOS + 3
assert MAX_CTX_ROWS == 23

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("=" * 120)
print("KS QUERY-t / FD2-LF72 -> SPECTRAL-HF256 / WITH-t / MIXED-M PAD23 -- LARGE MODEL 288-H6-FF576")
print("=" * 120)
print("DEVICE       :", DEVICE)
print("DATASET_ROOT :", DATASET_ROOT)
print("SAVE_DIR     :", SAVE_DIR)
print("TRAIN/TEST   :", f"{N_TRAIN_OP}/{N_TEST_OP} beta operators from NPZ saved split")
print("M train      :", "1..5, independent M per episode")
print("Episode      :", "same beta + same trajectory + M+1 distinct random eligible times")
print("Model input  :", "x / t / uc / uf; t explicitly supplied as constant normalized row")
print("Batching     :", "right-pad to 23 + src_key_padding_mask + true query gather")
print("Train monitor:", "fixed M=5, 512 train-side cases; SELECTS best checkpoint")
print(
    "Fixed test   :",
    f"M=5, {TEST_CASES_PER_OPERATOR}/operator x {N_TEST_OP} "
    f"= {TEST_CASES_PER_OPERATOR * N_TEST_OP} cases; diagnostic only",
)
print(
    "Formal test  :",
    f"independent M=5, {FORMAL_TEST_CASES} unseen-beta cases, BEST checkpoint only",
)
print("Epochs       :", EPOCHS)
print("=" * 120)


# ============================================================
# 1. LOCATE EXACT FORMAL DATASET
# ============================================================

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        "KS dataset root does not exist:\n"
        f"{DATASET_ROOT}"
    )

matches = sorted(
    DATASET_ROOT.rglob(EXPECTED_DATASET_NAME)
)

if len(matches) == 0:
    all_npz = sorted(DATASET_ROOT.rglob("*.npz"))
    found_text = "\n".join(f"  {p}" for p in all_npz)

    raise FileNotFoundError(
        "Could not find the formal KS NPZ under:\n"
        f"{DATASET_ROOT}\n\n"
        f"Expected filename:\n  {EXPECTED_DATASET_NAME}\n\n"
        "NPZ files actually found:\n"
        + (found_text if found_text else "  (none)")
    )

if len(matches) > 1:
    print("\nWARNING: multiple exact-name matches found:")
    for p in matches:
        print(" ", p)
    print("Using the first sorted match.")

DATASET_PATH = matches[0]
print("\nDATASET_PATH:", DATASET_PATH)


# ============================================================
# 2. LOAD MODEL-FACING DATA + EXACT SAVED SPLIT
# ============================================================

with np.load(DATASET_PATH, allow_pickle=False) as f:
    required = [
        "beta",
        "times",
        "eligible_time_idx",
        "eligible_times",
        "x",
        "uc",
        "uf",
        "train_op_idx",
        "test_op_idx",
        "T",
        "dt",
        "snapshot_dt",
        "nx_lf",
        "nx_hf",
        "nr",
        "n_traj",
        "n_time",
        "lf_solver_tag",
        "hf_solver_tag",
    ]

    for key in required:
        if key not in f:
            raise KeyError(
                f"Missing required NPZ key: {key}\n"
                f"Available keys: {f.files}"
            )

    beta_all = f["beta"][:].astype(np.float64).reshape(-1)
    times = f["times"][:].astype(np.float64).reshape(-1)

    ELIGIBLE_TIME_IDX = (
        f["eligible_time_idx"][:]
        .astype(np.int64)
        .reshape(-1)
    )

    eligible_times_saved = (
        f["eligible_times"][:]
        .astype(np.float64)
        .reshape(-1)
    )

    x_grid = f["x"][:].astype(np.float32).reshape(-1)

    # Model-facing common-grid arrays:
    # [Nop, Ntraj, Nt, Nr] = [200,20,71,100]
    uc_all = f["uc"][:].astype(np.float32)
    uf_all = f["uf"][:].astype(np.float32)

    train_ops = (
        f["train_op_idx"][:]
        .astype(np.int64)
        .reshape(-1)
    )

    test_ops = (
        f["test_op_idx"][:]
        .astype(np.int64)
        .reshape(-1)
    )

    nx_lf_saved = int(
        np.asarray(f["nx_lf"]).reshape(-1)[0]
    )

    nx_hf_saved = int(
        np.asarray(f["nx_hf"]).reshape(-1)[0]
    )

    nr_saved = int(
        np.asarray(f["nr"]).reshape(-1)[0]
    )

    T_saved = float(
        np.asarray(f["T"]).reshape(-1)[0]
    )

    dt_saved = float(
        np.asarray(f["dt"]).reshape(-1)[0]
    )

    snapshot_dt_saved = float(
        np.asarray(f["snapshot_dt"]).reshape(-1)[0]
    )

    n_traj_saved = int(
        np.asarray(f["n_traj"]).reshape(-1)[0]
    )

    n_time_saved = int(
        np.asarray(f["n_time"]).reshape(-1)[0]
    )

    lf_solver_tag_saved = str(
        np.asarray(f["lf_solver_tag"]).item()
    )

    hf_solver_tag_saved = str(
        np.asarray(f["hf_solver_tag"]).item()
    )

    dataset_metadata = {}
    if "metadata_json" in f:
        try:
            raw_meta = np.asarray(f["metadata_json"]).item()
            if isinstance(raw_meta, bytes):
                raw_meta = raw_meta.decode("utf-8")
            dataset_metadata = json.loads(str(raw_meta))
        except Exception as e:
            print("WARNING: could not parse metadata_json:", repr(e))


# ============================================================
# 3. INTEGRITY CHECKS
# ============================================================

if uc_all.ndim != 4:
    raise RuntimeError(
        f"Expected uc shape [Nop,Ntraj,Nt,Nr], got {uc_all.shape}"
    )

N_OP, N_TRAJ, N_TIME, NX = uc_all.shape

assert N_OP == N_OP_EXPECTED, N_OP
assert N_TRAJ == N_TRAJ_EXPECTED, N_TRAJ
assert N_TIME == N_TIME_EXPECTED, N_TIME
assert NX == NR_SHARED_EXPECTED, NX

assert uf_all.shape == (N_OP, N_TRAJ, N_TIME, NX)
assert beta_all.shape == (N_OP,)
assert times.shape == (N_TIME,)
assert x_grid.shape == (NX,)

assert len(train_ops) == N_TRAIN_OP
assert len(test_ops) == N_TEST_OP
assert len(np.unique(train_ops)) == N_TRAIN_OP
assert len(np.unique(test_ops)) == N_TEST_OP
assert len(np.intersect1d(train_ops, test_ops)) == 0

assert np.array_equal(
    np.sort(
        np.concatenate(
            [train_ops, test_ops]
        )
    ),
    np.arange(
        N_OP,
        dtype=np.int64,
    ),
)

assert np.all(np.isfinite(uc_all))
assert np.all(np.isfinite(uf_all))
assert np.all(np.isfinite(beta_all))
assert np.all(np.isfinite(times))
assert np.all(np.isfinite(x_grid))

assert np.isclose(beta_all.min(), 0.070, atol=1e-12)
assert np.isclose(beta_all.max(), 0.085, atol=1e-12)
assert np.all(np.diff(times) > 0.0)
assert np.isclose(times[0], 0.0, atol=1e-12)
assert np.isclose(times[-1], T_EXPECTED, atol=1e-12)

assert nx_lf_saved == NC_NATIVE_EXPECTED, nx_lf_saved
assert nx_hf_saved == NF_NATIVE_EXPECTED, nx_hf_saved
assert nr_saved == NR_SHARED_EXPECTED, nr_saved
assert np.isclose(T_saved, T_EXPECTED, atol=1e-12), T_saved
assert np.isclose(
    snapshot_dt_saved,
    SNAPSHOT_DT_EXPECTED,
    atol=1e-12,
), snapshot_dt_saved
assert n_traj_saved == N_TRAJ_EXPECTED, n_traj_saved
assert n_time_saved == N_TIME_EXPECTED, n_time_saved

assert lf_solver_tag_saved == LF_SOLVER_TAG_EXPECTED, (
    lf_solver_tag_saved,
    LF_SOLVER_TAG_EXPECTED,
)

assert hf_solver_tag_saved == HF_SOLVER_TAG_EXPECTED, (
    hf_solver_tag_saved,
    HF_SOLVER_TAG_EXPECTED,
)

N_ELIGIBLE_TIME = int(
    ELIGIBLE_TIME_IDX.size
)

assert N_ELIGIBLE_TIME == N_ELIGIBLE_TIME_EXPECTED, (
    N_ELIGIBLE_TIME,
    ELIGIBLE_TIME_IDX,
)

assert np.all(ELIGIBLE_TIME_IDX >= 0)
assert np.all(ELIGIBLE_TIME_IDX < N_TIME)
assert len(np.unique(ELIGIBLE_TIME_IDX)) == N_ELIGIBLE_TIME
assert np.all(np.diff(ELIGIBLE_TIME_IDX) > 0)

eligible_times = times[
    ELIGIBLE_TIME_IDX
]

assert np.allclose(
    eligible_times,
    eligible_times_saved,
    atol=1e-12,
    rtol=0.0,
)

assert np.isclose(
    eligible_times[0],
    MIN_ELIGIBLE_T_EXPECTED,
    atol=1e-12,
)

assert np.isclose(
    eligible_times[-1],
    MAX_ELIGIBLE_T_EXPECTED,
    atol=1e-12,
)

expected_eligible = np.arange(
    MIN_ELIGIBLE_T_EXPECTED,
    MAX_ELIGIBLE_T_EXPECTED + 0.5 * SNAPSHOT_DT_EXPECTED,
    SNAPSHOT_DT_EXPECTED,
    dtype=np.float64,
)

assert np.allclose(
    eligible_times,
    expected_eligible,
    atol=1e-12,
    rtol=0.0,
)

print("\nDataset:")
print("  beta shape       :", beta_all.shape)
print("  times shape      :", times.shape)
print("  x shape          :", x_grid.shape)
print("  uc shape         :", uc_all.shape)
print("  uf shape         :", uf_all.shape)
print("  beta range       :", f"[{beta_all.min():.8f}, {beta_all.max():.8f}]")
print("  native LF/HF     :", nx_lf_saved, "/", nx_hf_saved)
print("  LF solver tag    :", lf_solver_tag_saved)
print("  HF solver tag    :", hf_solver_tag_saved)
print("  shared Nr        :", NX)
print("  T / solver dt    :", T_saved, "/", dt_saved)
print("  snapshot dt      :", snapshot_dt_saved)
print("  trajectories/op  :", N_TRAJ)
print("  saved times      :", N_TIME)
print(
    "  eligible times   :",
    N_ELIGIBLE_TIME,
    f"({eligible_times[0]:.2f}..{eligible_times[-1]:.2f})",
)
print("  train/test beta  :", len(train_ops), "/", len(test_ops))
print(
    "  task             : same-beta, same-(beta,u0)-trajectory "
    "random-time ICL; t explicitly supplied to model"
)


# ============================================================
# 4. SAVE OPERATOR SPLIT TABLE
# ============================================================

split_df = pd.DataFrame({
    "operator": np.arange(N_OP, dtype=np.int64),
    "beta": beta_all,
    "split": np.where(
        np.isin(
            np.arange(N_OP),
            train_ops,
        ),
        "train",
        "test",
    ),
})

split_df.to_csv(
    SAVE_DIR / "operator_split.csv",
    index=False,
)

np.savez(
    SAVE_DIR / "operator_split.npz",
    train_ops=train_ops,
    test_ops=test_ops,
    beta=beta_all,
)


# ============================================================
# 5. TRAIN-ONLY NORMALIZATION
# ============================================================

# uc/uf statistics use ONLY training beta operators
# and ONLY the eligible late-time pool.
uc_train_phys = (
    uc_all[train_ops]
    [:, :, ELIGIBLE_TIME_IDX, :]
)

uf_train_phys = (
    uf_all[train_ops]
    [:, :, ELIGIBLE_TIME_IDX, :]
)

uc_mean = float(
    uc_train_phys.mean()
)

uc_std = float(
    uc_train_phys.std() + 1e-6
)

uf_mean = float(
    uf_train_phys.mean()
)

uf_std = float(
    uf_train_phys.std() + 1e-6
)

x_mean = float(
    x_grid.mean()
)

x_std = float(
    x_grid.std() + 1e-6
)

# t is a deterministic coordinate, normalized from
# the saved eligible-time grid itself.
t_mean = float(
    eligible_times.mean()
)

t_std = float(
    eligible_times.std() + 1e-6
)

x_n = (
    (x_grid - x_mean)
    / x_std
).astype(
    np.float32
)

uc_n = (
    (uc_all - uc_mean)
    / uc_std
).astype(
    np.float32
)

uf_n = (
    (uf_all - uf_mean)
    / uf_std
).astype(
    np.float32
)

time_scalar_n = (
    (times - t_mean)
    / t_std
).astype(
    np.float32
)

# Constant Nr-wide row carrying scalar time.
time_rows_n = np.broadcast_to(
    time_scalar_n[:, None],
    (
        N_TIME,
        NX,
    ),
).copy()

print("\nNormalization -- TRAIN beta operators, eligible times only:")
print(f"  x  mean/std : {x_mean:.8f} / {x_std:.8f}")
print(f"  t  mean/std : {t_mean:.8f} / {t_std:.8f}")
print(f"  uc mean/std : {uc_mean:.8f} / {uc_std:.8f}")
print(f"  uf mean/std : {uf_mean:.8f} / {uf_std:.8f}")


# ============================================================
# 6. RAW LF -> HF BASELINE
# ============================================================

def rel_l2_np(
    pred,
    target,
):
    num = np.linalg.norm(
        pred - target,
        axis=-1,
    )

    den = np.linalg.norm(
        target,
        axis=-1,
    )

    return (
        num
        / np.maximum(
            den,
            EPS_METRIC,
        )
    )


def summarize_array(
    arr,
    scale=1.0,
):
    arr = (
        np.asarray(
            arr,
            dtype=np.float64,
        )
        * scale
    )

    return {
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "std": float(np.std(arr)),
        "p90": float(np.quantile(arr, 0.90)),
        "p95": float(np.quantile(arr, 0.95)),
        "p99": float(np.quantile(arr, 0.99)),
    }


def short_summary_str(
    d,
):
    return (
        f"min={d['min']:.3f}, "
        f"max={d['max']:.3f}, "
        f"mean={d['mean']:.3f}, "
        f"median={d['median']:.3f}, "
        f"p90={d['p90']:.3f}, "
        f"std={d['std']:.3f}"
    )


train_raw_rel = rel_l2_np(
    uc_all[train_ops]
    [:, :, ELIGIBLE_TIME_IDX, :],
    uf_all[train_ops]
    [:, :, ELIGIBLE_TIME_IDX, :],
).reshape(-1)

test_raw_rel_full = rel_l2_np(
    uc_all[test_ops]
    [:, :, ELIGIBLE_TIME_IDX, :],
    uf_all[test_ops]
    [:, :, ELIGIBLE_TIME_IDX, :],
).reshape(-1)

print("\nRaw uc -> uf baseline over eligible trajectory-time pool:")
print(
    "  TRAIN:",
    short_summary_str(
        summarize_array(
            train_raw_rel,
            scale=100.0,
        )
    ),
)

print(
    "  TEST :",
    short_summary_str(
        summarize_array(
            test_raw_rel_full,
            scale=100.0,
        )
    ),
)


# ============================================================
# 7. MIXED-M PAD23 TRAIN BATCHING
# ============================================================

rng_train = np.random.default_rng(
    SEED + 100
)


def sample_distinct_time_indices(
    rng,
    batch_size,
    n_pick,
):
    """
    Return [batch_size,n_pick] distinct GLOBAL
    eligible-time indices per row.
    """
    local = rng.integers(
        0,
        N_ELIGIBLE_TIME,
        size=(
            batch_size,
            n_pick,
        ),
        dtype=np.int64,
    )

    while True:
        sorted_local = np.sort(
            local,
            axis=1,
        )

        bad = np.any(
            sorted_local[:, 1:]
            == sorted_local[:, :-1],
            axis=1,
        )

        if not np.any(bad):
            break

        n_bad = int(
            np.sum(bad)
        )

        local[bad] = rng.integers(
            0,
            N_ELIGIBLE_TIME,
            size=(
                n_bad,
                n_pick,
            ),
            dtype=np.int64,
        )

    return ELIGIBLE_TIME_IDX[
        local
    ]


def row_types_for_m(
    m,
):
    types = []

    for _ in range(m):
        types.extend([
            TYPE_X_DEMO,
            TYPE_T_DEMO,
            TYPE_UC_DEMO,
            TYPE_UF_DEMO,
        ])

    types.extend([
        TYPE_X_QUERY,
        TYPE_T_QUERY,
        TYPE_UC_QUERY,
    ])

    return np.asarray(
        types,
        dtype=np.int64,
    )


ROW_TYPES_BY_M = {
    m: row_types_for_m(m)
    for m in range(
        MIN_DEMOS,
        MAX_DEMOS + 1,
    )
}


def make_train_batch(
    batch_size,
):
    """
    One training episode:
      1) draw one TRAIN beta operator;
      2) draw one trajectory under that beta;
      3) draw M ~ Uniform{1,...,5};
      4) draw M+1 DISTINCT eligible time slots;
      5) first M random-order slots are demos,
         last slot is query.

    Demo rows:
        [x, t_d, uc(t_d), uf(t_d)]

    Query rows:
        [x, t_q, uc(t_q)]

    Target:
        uf(t_q) directly.
    """
    S_PAD = MAX_CTX_ROWS

    m_arr = rng_train.integers(
        MIN_DEMOS,
        MAX_DEMOS + 1,
        size=batch_size,
        dtype=np.int64,
    )

    op_pos = rng_train.integers(
        0,
        len(train_ops),
        size=batch_size,
        dtype=np.int64,
    )

    op_batch = train_ops[
        op_pos
    ]

    traj_batch = rng_train.integers(
        0,
        N_TRAJ,
        size=batch_size,
        dtype=np.int64,
    )

    mat_ctx = np.zeros(
        (
            batch_size,
            S_PAD,
            NX,
        ),
        dtype=np.float32,
    )

    row_types = np.zeros(
        (
            batch_size,
            S_PAD,
        ),
        dtype=np.int64,
    )

    padding_mask = np.ones(
        (
            batch_size,
            S_PAD,
        ),
        dtype=np.bool_,
    )

    query_pos = np.empty(
        batch_size,
        dtype=np.int64,
    )

    target_n = np.empty(
        (
            batch_size,
            NX,
        ),
        dtype=np.float32,
    )

    for m in range(
        MIN_DEMOS,
        MAX_DEMOS + 1,
    ):
        idx = np.flatnonzero(
            m_arr == m
        )

        Bm = int(
            idx.size
        )

        if Bm == 0:
            continue

        S = (
            4 * m
            + 3
        )

        ops_m = op_batch[
            idx
        ]

        traj_m = traj_batch[
            idx
        ]

        time_slots = (
            sample_distinct_time_indices(
                rng_train,
                batch_size=Bm,
                n_pick=m + 1,
            )
        )

        demo_times = time_slots[
            :,
            :m,
        ]

        query_times = time_slots[
            :,
            m,
        ]

        row_types[
            idx,
            :S,
        ] = ROW_TYPES_BY_M[
            m
        ][None, :]

        padding_mask[
            idx,
            :S,
        ] = False

        row = 0

        for j in range(m):
            mat_ctx[
                idx,
                row,
                :,
            ] = x_n[
                None,
                :,
            ]

            row += 1

            mat_ctx[
                idx,
                row,
                :,
            ] = time_rows_n[
                demo_times[:, j],
                :,
            ]

            row += 1

            mat_ctx[
                idx,
                row,
                :,
            ] = uc_n[
                ops_m,
                traj_m,
                demo_times[:, j],
                :,
            ]

            row += 1

            mat_ctx[
                idx,
                row,
                :,
            ] = uf_n[
                ops_m,
                traj_m,
                demo_times[:, j],
                :,
            ]

            row += 1

        mat_ctx[
            idx,
            row,
            :,
        ] = x_n[
            None,
            :,
        ]

        row += 1

        mat_ctx[
            idx,
            row,
            :,
        ] = time_rows_n[
            query_times,
            :,
        ]

        row += 1

        mat_ctx[
            idx,
            row,
            :,
        ] = uc_n[
            ops_m,
            traj_m,
            query_times,
            :,
        ]

        row += 1

        assert row == S

        query_pos[
            idx
        ] = (
            S - 1
        )

        target_n[
            idx
        ] = uf_n[
            ops_m,
            traj_m,
            query_times,
            :,
        ]

    assert np.all(
        ~padding_mask[
            np.arange(
                batch_size
            ),
            query_pos,
        ]
    )

    return (
        m_arr,
        torch.from_numpy(
            mat_ctx
        ).to(
            DEVICE,
            non_blocking=True,
        ),
        torch.from_numpy(
            row_types
        ).to(
            DEVICE,
            non_blocking=True,
        ),
        torch.from_numpy(
            padding_mask
        ).to(
            DEVICE,
            non_blocking=True,
        ),
        torch.from_numpy(
            query_pos
        ).to(
            DEVICE,
            non_blocking=True,
        ),
        torch.from_numpy(
            target_n
        ).to(
            DEVICE,
            non_blocking=True,
        ),
    )


# ============================================================
# 8. FIXED MONITOR / TEST CASES
# ============================================================

# Case tuple:
#   (operator, trajectory, demo_time_indices[M], query_time_index)


def make_fixed_cases(
    op_pool,
    n_cases,
    seed,
    fixed_m,
):
    r = np.random.default_rng(
        seed
    )

    cases = []

    for _ in range(
        n_cases
    ):
        op = int(
            r.choice(
                op_pool
            )
        )

        traj = int(
            r.integers(
                0,
                N_TRAJ,
            )
        )

        slots = r.choice(
            ELIGIBLE_TIME_IDX,
            size=fixed_m + 1,
            replace=False,
        ).astype(
            np.int64
        )

        cases.append(
            (
                op,
                traj,
                slots[:fixed_m],
                int(
                    slots[-1]
                ),
            )
        )

    return cases


def make_operator_balanced_fixed_cases(
    op_pool,
    cases_per_operator,
    seed,
    fixed_m,
):
    """
    Every unseen beta contributes exactly cases_per_operator cases.

    Within each operator, (trajectory, query-time) pairs are unique.
    Every case's M demos come from the same trajectory as the query.
    """
    r = np.random.default_rng(
        seed
    )

    cases = []

    flat_candidates = np.arange(
        N_TRAJ
        * N_ELIGIBLE_TIME,
        dtype=np.int64,
    )

    for op in np.asarray(
        op_pool,
        dtype=np.int64,
    ):
        chosen = r.choice(
            flat_candidates,
            size=cases_per_operator,
            replace=False,
        )

        for flat in chosen:
            traj = int(
                flat
                // N_ELIGIBLE_TIME
            )

            q_local = int(
                flat
                % N_ELIGIBLE_TIME
            )

            q_time_idx = int(
                ELIGIBLE_TIME_IDX[
                    q_local
                ]
            )

            candidates = ELIGIBLE_TIME_IDX[
                ELIGIBLE_TIME_IDX
                != q_time_idx
            ]

            demo_time_idx = r.choice(
                candidates,
                size=fixed_m,
                replace=False,
            ).astype(
                np.int64
            )

            cases.append(
                (
                    int(op),
                    traj,
                    demo_time_idx,
                    q_time_idx,
                )
            )

    return cases


def make_independent_formal_test_cases(
    op_pool,
    n_cases,
    seed,
    fixed_m,
):
    r = np.random.default_rng(
        seed
    )

    cases = []

    for _ in range(
        n_cases
    ):
        op = int(
            r.choice(
                op_pool
            )
        )

        traj = int(
            r.integers(
                0,
                N_TRAJ,
            )
        )

        slots = r.choice(
            ELIGIBLE_TIME_IDX,
            size=fixed_m + 1,
            replace=False,
        ).astype(
            np.int64
        )

        cases.append(
            (
                op,
                traj,
                slots[:fixed_m],
                int(
                    slots[-1]
                ),
            )
        )

    return cases


monitor_cases = make_fixed_cases(
    train_ops,
    n_cases=N_MONITOR_CASES,
    seed=SEED + 201,
    fixed_m=TEST_FIXED_DEMOS,
)

fixed_test_cases = (
    make_operator_balanced_fixed_cases(
        test_ops,
        cases_per_operator=
            TEST_CASES_PER_OPERATOR,
        seed=TEST_CASE_SEED,
        fixed_m=TEST_FIXED_DEMOS,
    )
)

assert len(
    fixed_test_cases
) == (
    N_TEST_OP
    * TEST_CASES_PER_OPERATOR
)

assert set(
    c[0]
    for c in fixed_test_cases
) == set(
    int(x)
    for x in test_ops
)

formal_test_cases = (
    make_independent_formal_test_cases(
        test_ops,
        n_cases=FORMAL_TEST_CASES,
        seed=FORMAL_TEST_CASE_SEED,
        fixed_m=TEST_FIXED_DEMOS,
    )
)

assert len(
    formal_test_cases
) == FORMAL_TEST_CASES


def save_case_table(
    cases,
    path,
):
    rows = []

    for case_id, (
        op,
        traj,
        demo_time_idx,
        q_time_idx,
    ) in enumerate(
        cases
    ):
        rows.append({
            "case_id":
                case_id,
            "operator":
                int(op),
            "beta":
                float(
                    beta_all[
                        op
                    ]
                ),
            "trajectory":
                int(
                    traj
                ),
            "demo_time_indices":
                ",".join(
                    str(
                        int(x)
                    )
                    for x in demo_time_idx
                ),
            "demo_times":
                ",".join(
                    f"{times[int(x)]:.2f}"
                    for x in demo_time_idx
                ),
            "query_time_index":
                int(
                    q_time_idx
                ),
            "query_time":
                float(
                    times[
                        q_time_idx
                    ]
                ),
        })

    pd.DataFrame(
        rows
    ).to_csv(
        path,
        index=False,
    )


save_case_table(
    fixed_test_cases,
    SAVE_DIR
    / "fixed_unseen_test_cases.csv",
)

save_case_table(
    formal_test_cases,
    SAVE_DIR
    / "formal_test_1024_cases.csv",
)


def make_eval_batch(
    cases_subset,
):
    batch_size = len(
        cases_subset
    )

    m = len(
        cases_subset[
            0
        ][
            2
        ]
    )

    S = (
        4 * m
        + 3
    )

    mat_ctx = np.empty(
        (
            batch_size,
            S,
            NX,
        ),
        dtype=np.float32,
    )

    row_types = np.broadcast_to(
        ROW_TYPES_BY_M[
            m
        ][
            None,
            :,
        ],
        (
            batch_size,
            S,
        ),
    ).copy()

    target = np.empty(
        (
            batch_size,
            NX,
        ),
        dtype=np.float32,
    )

    input_uc = np.empty(
        (
            batch_size,
            NX,
        ),
        dtype=np.float32,
    )

    op_arr = np.empty(
        batch_size,
        dtype=np.int64,
    )

    traj_arr = np.empty(
        batch_size,
        dtype=np.int64,
    )

    qtime_idx_arr = np.empty(
        batch_size,
        dtype=np.int64,
    )

    for i, (
        op,
        traj,
        demo_time_idx,
        q_time_idx,
    ) in enumerate(
        cases_subset
    ):
        row = 0

        for ti in demo_time_idx:
            mat_ctx[
                i,
                row,
                :,
            ] = x_n

            row += 1

            mat_ctx[
                i,
                row,
                :,
            ] = time_rows_n[
                ti,
                :,
            ]

            row += 1

            mat_ctx[
                i,
                row,
                :,
            ] = uc_n[
                op,
                traj,
                ti,
                :,
            ]

            row += 1

            mat_ctx[
                i,
                row,
                :,
            ] = uf_n[
                op,
                traj,
                ti,
                :,
            ]

            row += 1

        mat_ctx[
            i,
            row,
            :,
        ] = x_n

        row += 1

        mat_ctx[
            i,
            row,
            :,
        ] = time_rows_n[
            q_time_idx,
            :,
        ]

        row += 1

        mat_ctx[
            i,
            row,
            :,
        ] = uc_n[
            op,
            traj,
            q_time_idx,
            :,
        ]

        row += 1

        assert row == S

        target[
            i
        ] = uf_all[
            op,
            traj,
            q_time_idx,
            :,
        ]

        input_uc[
            i
        ] = uc_all[
            op,
            traj,
            q_time_idx,
            :,
        ]

        op_arr[
            i
        ] = op

        traj_arr[
            i
        ] = traj

        qtime_idx_arr[
            i
        ] = q_time_idx

    return {
        "mat_ctx":
            torch.from_numpy(
                mat_ctx
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "row_types":
            torch.from_numpy(
                row_types
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "target":
            torch.from_numpy(
                target
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "input_uc":
            torch.from_numpy(
                input_uc
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "op":
            op_arr,

        "traj":
            traj_arr,

        "qtime_idx":
            qtime_idx_arr,
    }


# ============================================================
# 9. MODEL
# ============================================================

class RowSequenceWholeBlockShiftTransformer(
    nn.Module
):
    def __init__(
        self,
        row_width,
        max_ctx_rows=23,
        n_row_types=7,
        d_model=256,
        nhead=4,
        num_layers=5,
        dim_ff=512,
        dropout=0.1,
    ):
        super().__init__()

        self.row_in = nn.Linear(
            row_width,
            d_model,
        )

        self.row_type_emb = nn.Embedding(
            n_row_types,
            d_model,
        )

        self.pos_emb = nn.Embedding(
            max_ctx_rows,
            d_model,
        )

        enc_layer = (
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_ff,
                dropout=dropout,
                activation=torch.tanh,
                batch_first=True,
                norm_first=True,
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                enc_layer,
                num_layers=num_layers,
            )
        )

        self.row_out = nn.Sequential(
            nn.LayerNorm(
                d_model
            ),
            nn.Linear(
                d_model,
                d_model,
            ),
            nn.Tanh(),
            nn.Linear(
                d_model,
                row_width,
            ),
        )

        pos_ids = torch.arange(
            max_ctx_rows,
            dtype=torch.long,
        )

        self.register_buffer(
            "pos_ids",
            pos_ids,
            persistent=False,
        )

    def forward(
        self,
        mat_ctx,
        row_type_arr,
        key_padding_mask=None,
    ):
        B, S, _ = mat_ctx.shape

        if S > MAX_CTX_ROWS:
            raise ValueError(
                f"Sequence length {S} exceeds "
                f"MAX_CTX_ROWS={MAX_CTX_ROWS}"
            )

        if key_padding_mask is not None:
            if key_padding_mask.shape != (
                B,
                S,
            ):
                raise ValueError(
                    "Bad key_padding_mask shape: "
                    f"{tuple(key_padding_mask.shape)} "
                    f"vs {(B, S)}"
                )

            if key_padding_mask.dtype != torch.bool:
                raise TypeError(
                    "key_padding_mask must be torch.bool."
                )

        h = self.row_in(
            mat_ctx
        )

        h = (
            h
            + self.row_type_emb(
                row_type_arr
            )
            + self.pos_emb(
                self.pos_ids[
                    :S
                ]
            )[
                None,
                :,
                :,
            ]
        )

        h = self.encoder(
            h,
            src_key_padding_mask=
                key_padding_mask,
        )

        return self.row_out(
            h
        )


model = (
    RowSequenceWholeBlockShiftTransformer(
        row_width=NX,
        max_ctx_rows=MAX_CTX_ROWS,
        n_row_types=N_ROW_TYPES,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=D_FF,
        dropout=DROPOUT,
    )
    .to(
        DEVICE
    )
)

n_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nModel:")
print("  row width Nr    :", NX)
print("  d_model         :", D_MODEL)
print("  heads           :", N_HEAD)
print("  layers          :", N_LAYERS)
print("  FF              :", D_FF)
print("  dropout         :", DROPOUT)
print("  trainable params:", f"{n_params:,}")

assert n_params == EXPECTED_PARAMS, (
    n_params,
    EXPECTED_PARAMS,
)


# ============================================================
# 10. PADDING-INFORMATION SANITY CHECK
# ============================================================

@torch.no_grad()
def padding_information_sanity_check():
    model.eval()

    r = np.random.default_rng(
        SEED + 9999
    )

    m = 2
    S = (
        4 * m
        + 3
    )

    op = int(
        train_ops[
            0
        ]
    )

    traj = 0

    slots = r.choice(
        ELIGIBLE_TIME_IDX,
        size=m + 1,
        replace=False,
    ).astype(
        np.int64
    )

    demos = slots[
        :m
    ]

    q_time_idx = int(
        slots[
            -1
        ]
    )

    ctx = np.zeros(
        (
            1,
            MAX_CTX_ROWS,
            NX,
        ),
        dtype=np.float32,
    )

    types = np.zeros(
        (
            1,
            MAX_CTX_ROWS,
        ),
        dtype=np.int64,
    )

    mask = np.ones(
        (
            1,
            MAX_CTX_ROWS,
        ),
        dtype=np.bool_,
    )

    row = 0

    for ti in demos:
        ctx[
            0,
            row,
            :,
        ] = x_n
        types[
            0,
            row,
        ] = TYPE_X_DEMO
        row += 1

        ctx[
            0,
            row,
            :,
        ] = time_rows_n[
            ti,
            :,
        ]
        types[
            0,
            row,
        ] = TYPE_T_DEMO
        row += 1

        ctx[
            0,
            row,
            :,
        ] = uc_n[
            op,
            traj,
            ti,
            :,
        ]
        types[
            0,
            row,
        ] = TYPE_UC_DEMO
        row += 1

        ctx[
            0,
            row,
            :,
        ] = uf_n[
            op,
            traj,
            ti,
            :,
        ]
        types[
            0,
            row,
        ] = TYPE_UF_DEMO
        row += 1

    ctx[
        0,
        row,
        :,
    ] = x_n
    types[
        0,
        row,
    ] = TYPE_X_QUERY
    row += 1

    ctx[
        0,
        row,
        :,
    ] = time_rows_n[
        q_time_idx,
        :,
    ]
    types[
        0,
        row,
    ] = TYPE_T_QUERY
    row += 1

    ctx[
        0,
        row,
        :,
    ] = uc_n[
        op,
        traj,
        q_time_idx,
        :,
    ]
    types[
        0,
        row,
    ] = TYPE_UC_QUERY
    row += 1

    assert row == S

    mask[
        0,
        :S,
    ] = False

    query_pos = (
        S - 1
    )

    ctx_zero = ctx.copy()
    ctx_noise = ctx.copy()

    ctx_noise[
        0,
        S:,
        :,
    ] = r.normal(
        loc=50.0,
        scale=20.0,
        size=(
            MAX_CTX_ROWS - S,
            NX,
        ),
    ).astype(
        np.float32
    )

    types_noise = types.copy()

    types_noise[
        0,
        S:,
    ] = r.integers(
        0,
        N_ROW_TYPES,
        size=(
            MAX_CTX_ROWS - S
        ),
        dtype=np.int64,
    )

    mask_t = torch.from_numpy(
        mask
    ).to(
        DEVICE
    )

    out_zero = model(
        torch.from_numpy(
            ctx_zero
        ).to(
            DEVICE
        ),
        torch.from_numpy(
            types
        ).to(
            DEVICE
        ),
        key_padding_mask=
            mask_t,
    )

    out_noise = model(
        torch.from_numpy(
            ctx_noise
        ).to(
            DEVICE
        ),
        torch.from_numpy(
            types_noise
        ).to(
            DEVICE
        ),
        key_padding_mask=
            mask_t,
    )

    max_abs_diff = float(
        torch.max(
            torch.abs(
                out_zero[
                    0,
                    query_pos,
                    :,
                ]
                - out_noise[
                    0,
                    query_pos,
                    :,
                ]
            )
        )
        .detach()
        .cpu()
    )

    print("\nPadding-information sanity check:")
    print("  M                 :", m)
    print("  valid length      :", S)
    print(
        "  max |predA-predB| :",
        f"{max_abs_diff:.3e}",
    )

    if max_abs_diff > 1e-5:
        raise RuntimeError(
            "Padding sanity check failed."
        )

    print(
        "  PASS: masked padded rows are not prediction information."
    )

    model.train()


padding_information_sanity_check()


# ============================================================
# 11. OPTIMIZER / SCHEDULER
# ============================================================

TOTAL_STEPS = (
    EPOCHS
    * STEPS_PER_EPOCH
)

WARMUP_STEPS = max(
    1,
    int(
        WARMUP_FRAC
        * TOTAL_STEPS
    ),
)

opt = torch.optim.AdamW(
    model.parameters(),
    lr=MAX_LR,
    weight_decay=WEIGHT_DECAY,
)

warmup_scheduler = (
    torch.optim.lr_scheduler.LinearLR(
        opt,
        start_factor=(
            BASE_LR
            / MAX_LR
        ),
        end_factor=1.0,
        total_iters=WARMUP_STEPS,
    )
)

cosine_scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=max(
            1,
            TOTAL_STEPS
            - WARMUP_STEPS,
        ),
        eta_min=MIN_LR,
    )
)

scheduler = (
    torch.optim.lr_scheduler.SequentialLR(
        opt,
        schedulers=[
            warmup_scheduler,
            cosine_scheduler,
        ],
        milestones=[
            WARMUP_STEPS
        ],
    )
)

print("\nOptimizer/scheduler:")
print("  epochs      :", EPOCHS)
print("  total steps :", TOTAL_STEPS)
print("  warmup steps:", WARMUP_STEPS)
print("  base LR     :", BASE_LR)
print("  max LR      :", MAX_LR)
print("  min LR      :", MIN_LR)


# ============================================================
# 12. EVALUATION
# ============================================================

@torch.no_grad()
def eval_fixed_cases(
    cases,
    operator_level=False,
    return_case_arrays=False,
):
    model.eval()

    mse_all = []
    rel_all = []
    r2_all = []
    raw_uc_rel_all = []

    op_all = []
    traj_all = []
    qtime_idx_all = []

    for start in range(
        0,
        len(cases),
        EVAL_BATCH_SIZE,
    ):
        sub = cases[
            start:
            start + EVAL_BATCH_SIZE
        ]

        batch = make_eval_batch(
            sub
        )

        pred_all_n = model(
            batch[
                "mat_ctx"
            ],
            batch[
                "row_types"
            ],
        )

        # Formal/fixed evaluation here is M=5,
        # so S=23 and the query-uc row is the last row.
        pred_n = pred_all_n[
            :,
            -1,
            :,
        ]

        pred = (
            pred_n
            * uf_std
            + uf_mean
        )

        target = batch[
            "target"
        ]

        input_uc = batch[
            "input_uc"
        ]

        mse_i = torch.mean(
            (
                pred
                - target
            ) ** 2,
            dim=1,
        )

        rel_i = (
            torch.linalg.norm(
                pred
                - target,
                dim=1,
            )
            /
            torch.linalg.norm(
                target,
                dim=1,
            ).clamp_min(
                EPS_METRIC
            )
        )

        raw_uc_rel_i = (
            torch.linalg.norm(
                input_uc
                - target,
                dim=1,
            )
            /
            torch.linalg.norm(
                target,
                dim=1,
            ).clamp_min(
                EPS_METRIC
            )
        )

        target_mean = torch.mean(
            target,
            dim=1,
            keepdim=True,
        )

        sse = torch.sum(
            (
                pred
                - target
            ) ** 2,
            dim=1,
        )

        sst = torch.sum(
            (
                target
                - target_mean
            ) ** 2,
            dim=1,
        ).clamp_min(
            EPS_METRIC
        )

        r2_i = (
            1.0
            - sse
            / sst
        )

        mse_all.append(
            mse_i.cpu().numpy()
        )

        rel_all.append(
            rel_i.cpu().numpy()
        )

        r2_all.append(
            r2_i.cpu().numpy()
        )

        raw_uc_rel_all.append(
            raw_uc_rel_i.cpu().numpy()
        )

        op_all.append(
            batch[
                "op"
            ]
        )

        traj_all.append(
            batch[
                "traj"
            ]
        )

        qtime_idx_all.append(
            batch[
                "qtime_idx"
            ]
        )

    mse_all = np.concatenate(
        mse_all
    )

    rel_all = np.concatenate(
        rel_all
    )

    r2_all = np.concatenate(
        r2_all
    )

    raw_uc_rel_all = np.concatenate(
        raw_uc_rel_all
    )

    op_all = np.concatenate(
        op_all
    )

    traj_all = np.concatenate(
        traj_all
    )

    qtime_idx_all = np.concatenate(
        qtime_idx_all
    )

    result = {
        "MSE_mean":
            float(
                np.mean(
                    mse_all
                )
            ),

        "RelL2_summary_%":
            summarize_array(
                rel_all,
                scale=100.0,
            ),

        "R2_summary":
            summarize_array(
                r2_all
            ),

        "RawUcRelL2_summary_%":
            summarize_array(
                raw_uc_rel_all,
                scale=100.0,
            ),
    }

    if operator_level:
        op_mean_rel = []
        op_rows = []

        for op in np.unique(
            op_all
        ):
            mask = (
                op_all
                == op
            )

            vals = rel_all[
                mask
            ]

            raw_vals = raw_uc_rel_all[
                mask
            ]

            r2_vals = r2_all[
                mask
            ]

            op_mean_rel.append(
                float(
                    100.0
                    * np.mean(
                        vals
                    )
                )
            )

            op_rows.append({
                "operator":
                    int(op),
                "beta":
                    float(
                        beta_all[
                            op
                        ]
                    ),
                "n_cases":
                    int(
                        np.sum(
                            mask
                        )
                    ),
                "pred_mean_RelL2_pct":
                    float(
                        100.0
                        * np.mean(
                            vals
                        )
                    ),
                "pred_median_RelL2_pct":
                    float(
                        100.0
                        * np.median(
                            vals
                        )
                    ),
                "raw_mean_RelL2_pct":
                    float(
                        100.0
                        * np.mean(
                            raw_vals
                        )
                    ),
                "R2_mean":
                    float(
                        np.mean(
                            r2_vals
                        )
                    ),
            })

        result[
            "OperatorMeanRelL2_summary_%"
        ] = summarize_array(
            op_mean_rel
        )

        result[
            "operator_rows"
        ] = op_rows

    if return_case_arrays:
        result[
            "_case_arrays"
        ] = {
            "rel":
                rel_all,
            "raw_rel":
                raw_uc_rel_all,
            "r2":
                r2_all,
            "op":
                op_all,
            "traj":
                traj_all,
            "qtime_idx":
                qtime_idx_all,
            "qtime":
                times[
                    qtime_idx_all
                ],
        }

    model.train()

    return result


# ============================================================
# 13. CHECKPOINT / HISTORY
# ============================================================

BEST_CKPT = (
    SAVE_DIR
    / f"{RUN_TAG}_best_trainmonitor_fullstate.pt"
)

LATEST_CKPT = (
    SAVE_DIR
    / f"{RUN_TAG}_latest_fullstate.pt"
)

FINAL_CKPT = (
    SAVE_DIR
    / f"{RUN_TAG}_final_fullstate.pt"
)

HISTORY_JSON = (
    SAVE_DIR
    / f"{RUN_TAG}_history.json"
)

HISTORY_CSV = (
    SAVE_DIR
    / f"{RUN_TAG}_history.csv"
)

TEST_TRAJECTORY_CSV = (
    SAVE_DIR
    / f"{RUN_TAG}_fixed_test_trajectory.csv"
)

FINAL_METRICS_JSON = (
    SAVE_DIR
    / f"{RUN_TAG}_training_summary.json"
)

FORMAL_TEST_JSON = (
    SAVE_DIR
    / f"{RUN_TAG}_formal_test_1024.json"
)

FORMAL_CASE_CSV = (
    SAVE_DIR
    / f"{RUN_TAG}_formal_test_1024_cases_results.csv"
)

FORMAL_OPERATOR_CSV = (
    SAVE_DIR
    / f"{RUN_TAG}_formal_test_operator_summary.csv"
)


def get_rng_state():
    return {
        "python":
            random.getstate(),

        "numpy_legacy":
            np.random.get_state(),

        "torch":
            torch.get_rng_state(),

        "cuda":
            (
                torch.cuda.get_rng_state_all()
                if DEVICE == "cuda"
                else None
            ),

        "rng_train_state":
            rng_train.bit_generator.state,
    }


def set_rng_state(
    state,
):
    if not state:
        return

    random.setstate(
        state[
            "python"
        ]
    )

    np.random.set_state(
        state[
            "numpy_legacy"
        ]
    )

    torch.set_rng_state(
        state[
            "torch"
        ]
    )

    if (
        DEVICE == "cuda"
        and state.get(
            "cuda"
        ) is not None
    ):
        torch.cuda.set_rng_state_all(
            state[
                "cuda"
            ]
        )

    rng_train.bit_generator.state = (
        state[
            "rng_train_state"
        ]
    )


def save_checkpoint(
    path,
    epoch,
    global_step,
    history,
    test_trajectory,
    best_score,
    best_epoch,
):
    checkpoint = {
        "model":
            model.state_dict(),

        "optimizer":
            opt.state_dict(),

        "scheduler":
            scheduler.state_dict(),

        "epoch":
            int(
                epoch
            ),

        "global_step":
            int(
                global_step
            ),

        "rng_state":
            get_rng_state(),

        "norm": {
            "x_mean":
                x_mean,
            "x_std":
                x_std,
            "t_mean":
                t_mean,
            "t_std":
                t_std,
            "uc_mean":
                uc_mean,
            "uc_std":
                uc_std,
            "uf_mean":
                uf_mean,
            "uf_std":
                uf_std,
        },

        "config": {
            "RUN_TAG":
                RUN_TAG,
            "DATASET_PATH":
                str(
                    DATASET_PATH
                ),
            "SEED":
                SEED,
            "N_OP":
                N_OP,
            "N_TRAJ":
                N_TRAJ,
            "N_TIME":
                N_TIME,
            "N_ELIGIBLE_TIME":
                N_ELIGIBLE_TIME,
            "N_TRAIN_OP":
                N_TRAIN_OP,
            "N_TEST_OP":
                N_TEST_OP,
            "NX":
                NX,
            "MIN_DEMOS":
                MIN_DEMOS,
            "MAX_DEMOS":
                MAX_DEMOS,
            "TEST_FIXED_DEMOS":
                TEST_FIXED_DEMOS,
            "MAX_CTX_ROWS":
                MAX_CTX_ROWS,
            "BATCH_SIZE":
                BATCH_SIZE,
            "EPOCHS":
                EPOCHS,
            "STEPS_PER_EPOCH":
                STEPS_PER_EPOCH,
            "TEST_EVERY":
                TEST_EVERY,
            "TEST_CASES_PER_OPERATOR":
                TEST_CASES_PER_OPERATOR,
            "FORMAL_TEST_CASES":
                FORMAL_TEST_CASES,
            "D_MODEL":
                D_MODEL,
            "N_HEAD":
                N_HEAD,
            "N_LAYERS":
                N_LAYERS,
            "D_FF":
                D_FF,
            "DROPOUT":
                DROPOUT,
            "N_PARAMS":
                n_params,
            "WEIGHT_DECAY":
                WEIGHT_DECAY,
            "BASE_LR":
                BASE_LR,
            "MAX_LR":
                MAX_LR,
            "MIN_LR":
                MIN_LR,
            "WARMUP_FRAC":
                WARMUP_FRAC,
            "GRAD_CLIP":
                GRAD_CLIP,
            "train_ops":
                train_ops.tolist(),
            "test_ops":
                test_ops.tolist(),
            "operator_parameter":
                "beta",
            "beta_range": [
                float(
                    beta_all.min()
                ),
                float(
                    beta_all.max()
                ),
            ],
            "native_LF_HF": [
                nx_lf_saved,
                nx_hf_saved,
            ],
            "lf_solver_tag":
                lf_solver_tag_saved,
            "hf_solver_tag":
                hf_solver_tag_saved,
            "T":
                T_saved,
            "dt":
                dt_saved,
            "snapshot_dt":
                snapshot_dt_saved,
            "sample_axis":
                "time_along_fixed_beta_u0_trajectory",
            "trajectory_identity":
                "fixed_(beta,u0)",
            "eligible_time_range": [
                float(
                    eligible_times[
                        0
                    ]
                ),
                float(
                    eligible_times[
                        -1
                    ]
                ),
            ],
            "episode_rule":
                "same_beta_same_trajectory_M_plus_1_distinct_random_time_slots",
            "operator_given_explicitly_to_model":
                False,
            "time_given_explicitly_to_model":
                True,
            "demo_time_order":
                "random_sample_order_not_chronologically_sorted",
            "sequence":
                "[x,t_d1,uc(td1),uf(td1),...,x,t_dM,uc(tdM),uf(tdM),x,t_q,uc(tq)]->uf(tq)",
            "normalization":
                "x_uc_uf_scalar_stats; uc_uf from 140 train beta eligible times; t from eligible coordinate grid",
            "M_sampling":
                "independent_per_episode_uniform_1_to_5",
            "TRAIN_BATCHING":
                "mixed_M_independent_per_episode_pad_to_23",
            "PADDING_SIDE":
                "right",
            "PADDING_MASK":
                "src_key_padding_mask_bool_true_is_padding",
            "PREDICTION_POSITION":
                "final_valid_query_uc_row_4M_plus_2_zero_based",
            "checkpoint_selection":
                "fixed_M5_monitor_from_training_operators_only",
            "heldout_test_usage":
                "diagnostic_only_never_checkpoint_selection",
            "fixed_test_protocol":
                "60_unseen_beta_operators_x_16_same_trajectory_random_time_cases_per_operator_fixed_M5",
            "formal_test_protocol":
                "independent_1024_same_trajectory_random_time_M5_cases_from_60_heldout_beta_operators_after_training_only",
        },

        "dataset_metadata":
            dataset_metadata,

        "history":
            history,

        "test_trajectory":
            test_trajectory,

        "best_score":
            float(
                best_score
            ),

        "best_epoch":
            int(
                best_epoch
            ),
    }

    torch.save(
        checkpoint,
        path,
    )


def torch_load_full(
    path,
):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


# ============================================================
# 14. OPTIONAL AUTO-RESUME
# ============================================================

history = []
test_trajectory = []

best_score = float(
    "inf"
)

best_epoch = -1
global_step = 0
start_epoch = 1

if LATEST_CKPT.exists():
    print("\nFound latest checkpoint; auto-resuming:")
    print(" ", LATEST_CKPT)

    ckpt = torch_load_full(
        LATEST_CKPT
    )

    model.load_state_dict(
        ckpt[
            "model"
        ]
    )

    opt.load_state_dict(
        ckpt[
            "optimizer"
        ]
    )

    scheduler.load_state_dict(
        ckpt[
            "scheduler"
        ]
    )

    global_step = int(
        ckpt[
            "global_step"
        ]
    )

    last_epoch = int(
        ckpt[
            "epoch"
        ]
    )

    start_epoch = (
        last_epoch
        + 1
    )

    history = list(
        ckpt.get(
            "history",
            [],
        )
    )

    test_trajectory = list(
        ckpt.get(
            "test_trajectory",
            [],
        )
    )

    best_score = float(
        ckpt.get(
            "best_score",
            float(
                "inf"
            ),
        )
    )

    best_epoch = int(
        ckpt.get(
            "best_epoch",
            -1,
        )
    )

    set_rng_state(
        ckpt.get(
            "rng_state",
            None,
        )
    )

    print(
        f"Resumed after epoch {last_epoch}; "
        f"next epoch={start_epoch}; "
        f"global_step={global_step}; "
        f"best_epoch={best_epoch}; "
        f"best_train_monitor_mean={best_score:.6f}%"
    )


# ============================================================
# 15. EPOCH-0 DIAGNOSTIC FOR FRESH RUN
# ============================================================

if start_epoch == 1:
    train_eval0 = eval_fixed_cases(
        monitor_cases,
        operator_level=False,
    )

    test_eval0 = eval_fixed_cases(
        fixed_test_cases,
        operator_level=True,
    )

    train_s0 = train_eval0[
        "RelL2_summary_%"
    ]

    test_s0 = test_eval0[
        "RelL2_summary_%"
    ]

    print("\nEpoch 000 diagnostics:")

    print(
        "  TRAIN monitor:",
        short_summary_str(
            train_s0
        ),
    )

    print(
        "  TEST heldout :",
        short_summary_str(
            test_s0
        ),
    )

    print(
        "  TEST raw LF  :",
        short_summary_str(
            test_eval0[
                "RawUcRelL2_summary_%"
            ]
        ),
    )

    history.append({
        "epoch":
            0,
        "train_loss_norm_MSE":
            None,
        "lr":
            float(
                opt.param_groups[
                    0
                ][
                    "lr"
                ]
            ),
        "train_monitor_mean_RelL2_pct":
            train_s0[
                "mean"
            ],
        "train_monitor_median_RelL2_pct":
            train_s0[
                "median"
            ],
        "train_monitor_p90_RelL2_pct":
            train_s0[
                "p90"
            ],
        "test_mean_RelL2_pct":
            test_s0[
                "mean"
            ],
        "test_median_RelL2_pct":
            test_s0[
                "median"
            ],
        "test_p90_RelL2_pct":
            test_s0[
                "p90"
            ],
        "test_R2_mean":
            test_eval0[
                "R2_summary"
            ][
                "mean"
            ],
    })

    test_trajectory.append({
        "epoch":
            0,
        "mean_RelL2_pct":
            test_s0[
                "mean"
            ],
        "median_RelL2_pct":
            test_s0[
                "median"
            ],
        "p90_RelL2_pct":
            test_s0[
                "p90"
            ],
        "max_RelL2_pct":
            test_s0[
                "max"
            ],
        "R2_mean":
            test_eval0[
                "R2_summary"
            ][
                "mean"
            ],
        "operator_mean_RelL2_mean_pct":
            test_eval0[
                "OperatorMeanRelL2_summary_%"
            ][
                "mean"
            ],
    })

    # Train-side monitor is the ONLY checkpoint-selection metric.
    best_score = float(
        train_s0[
            "mean"
        ]
    )

    best_epoch = 0

    save_checkpoint(
        BEST_CKPT,
        epoch=0,
        global_step=0,
        history=history,
        test_trajectory=
            test_trajectory,
        best_score=
            best_score,
        best_epoch=
            best_epoch,
    )


# ============================================================
# 16. TRAIN
# ============================================================

training_start = time.time()

for epoch in range(
    start_epoch,
    EPOCHS + 1,
):
    model.train()

    epoch_t0 = time.time()

    losses = []

    m_counts = {
        m: 0
        for m in range(
            MIN_DEMOS,
            MAX_DEMOS + 1,
        )
    }

    for _ in range(
        STEPS_PER_EPOCH
    ):
        (
            m_arr,
            mat_ctx,
            row_types,
            padding_mask,
            query_pos,
            target_n,
        ) = make_train_batch(
            BATCH_SIZE
        )

        for m in m_counts:
            m_counts[
                m
            ] += int(
                np.sum(
                    m_arr
                    == m
                )
            )

        opt.zero_grad(
            set_to_none=True
        )

        pred_all_n = model(
            mat_ctx,
            row_types,
            key_padding_mask=
                padding_mask,
        )

        batch_indices = torch.arange(
            BATCH_SIZE,
            device=DEVICE,
        )

        pred_n = pred_all_n[
            batch_indices,
            query_pos,
            :,
        ]

        loss = torch.mean(
            (
                pred_n
                - target_n
            ) ** 2
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP,
        )

        opt.step()
        scheduler.step()

        global_step += 1

        losses.append(
            float(
                loss.detach().cpu()
            )
        )

    train_loss = float(
        np.mean(
            losses
        )
    )

    lr_now = float(
        opt.param_groups[
            0
        ][
            "lr"
        ]
    )

    epoch_seconds = (
        time.time()
        - epoch_t0
    )

    do_eval = (
        epoch
        % PRINT_EVERY
        == 0
        or epoch
        == EPOCHS
    )

    row = {
        "epoch":
            int(
                epoch
            ),
        "train_loss_norm_MSE":
            train_loss,
        "lr":
            lr_now,
        "epoch_seconds":
            float(
                epoch_seconds
            ),
    }

    if do_eval:
        train_eval = eval_fixed_cases(
            monitor_cases,
            operator_level=False,
        )

        train_s = train_eval[
            "RelL2_summary_%"
        ]

        row.update({
            "train_monitor_mean_RelL2_pct":
                train_s[
                    "mean"
                ],
            "train_monitor_median_RelL2_pct":
                train_s[
                    "median"
                ],
            "train_monitor_p90_RelL2_pct":
                train_s[
                    "p90"
                ],
            "train_monitor_R2_mean":
                train_eval[
                    "R2_summary"
                ][
                    "mean"
                ],
        })

        # BEST checkpoint selection uses TRAIN monitor only.
        if (
            train_s[
                "mean"
            ]
            < best_score
        ):
            best_score = float(
                train_s[
                    "mean"
                ]
            )

            best_epoch = int(
                epoch
            )

            save_checkpoint(
                BEST_CKPT,
                epoch=epoch,
                global_step=
                    global_step,
                history=
                    history + [row],
                test_trajectory=
                    test_trajectory,
                best_score=
                    best_score,
                best_epoch=
                    best_epoch,
            )

        if (
            epoch
            % TEST_EVERY
            == 0
            or epoch
            == EPOCHS
        ):
            test_eval = eval_fixed_cases(
                fixed_test_cases,
                operator_level=True,
            )

            test_s = test_eval[
                "RelL2_summary_%"
            ]

            op_s = test_eval[
                "OperatorMeanRelL2_summary_%"
            ]

            row.update({
                "test_mean_RelL2_pct":
                    test_s[
                        "mean"
                    ],
                "test_median_RelL2_pct":
                    test_s[
                        "median"
                    ],
                "test_p90_RelL2_pct":
                    test_s[
                        "p90"
                    ],
                "test_max_RelL2_pct":
                    test_s[
                        "max"
                    ],
                "test_R2_mean":
                    test_eval[
                        "R2_summary"
                    ][
                        "mean"
                    ],
                "test_R2_median":
                    test_eval[
                        "R2_summary"
                    ][
                        "median"
                    ],
                "test_raw_mean_RelL2_pct":
                    test_eval[
                        "RawUcRelL2_summary_%"
                    ][
                        "mean"
                    ],
                "test_operator_mean_min_pct":
                    op_s[
                        "min"
                    ],
                "test_operator_mean_max_pct":
                    op_s[
                        "max"
                    ],
                "test_operator_mean_mean_pct":
                    op_s[
                        "mean"
                    ],
                "test_operator_mean_median_pct":
                    op_s[
                        "median"
                    ],
                "test_operator_mean_p90_pct":
                    op_s[
                        "p90"
                    ],
            })

            test_trajectory.append({
                "epoch":
                    int(
                        epoch
                    ),
                "mean_RelL2_pct":
                    test_s[
                        "mean"
                    ],
                "median_RelL2_pct":
                    test_s[
                        "median"
                    ],
                "p90_RelL2_pct":
                    test_s[
                        "p90"
                    ],
                "max_RelL2_pct":
                    test_s[
                        "max"
                    ],
                "R2_mean":
                    test_eval[
                        "R2_summary"
                    ][
                        "mean"
                    ],
                "R2_median":
                    test_eval[
                        "R2_summary"
                    ][
                        "median"
                    ],
                "operator_mean_min_pct":
                    op_s[
                        "min"
                    ],
                "operator_mean_max_pct":
                    op_s[
                        "max"
                    ],
                "operator_mean_mean_pct":
                    op_s[
                        "mean"
                    ],
                "operator_mean_median_pct":
                    op_s[
                        "median"
                    ],
                "operator_mean_p90_pct":
                    op_s[
                        "p90"
                    ],
            })

            print(
                f"\nEpoch {epoch:03d} | "
                f"loss={train_loss:.6e} | "
                f"lr={lr_now:.3e} | "
                f"{epoch_seconds:.1f}s"
            )

            print(
                "  TRAIN monitor RelL2%:",
                short_summary_str(
                    train_s
                ),
            )

            print(
                "  TEST heldout RelL2% :",
                short_summary_str(
                    test_s
                ),
            )

            print(
                "  TEST R2             :",
                short_summary_str(
                    test_eval[
                        "R2_summary"
                    ]
                ),
            )

            print(
                "  TEST op-mean RelL2% :",
                short_summary_str(
                    op_s
                ),
            )

            print(
                f"  BEST train monitor  : "
                f"{best_score:.4f}% @ epoch {best_epoch}"
            )

            print(
                "  M counts this epoch :",
                m_counts,
            )

    history.append(
        row
    )

    # Save latest full state every epoch for restart safety.
    save_checkpoint(
        LATEST_CKPT,
        epoch=epoch,
        global_step=
            global_step,
        history=
            history,
        test_trajectory=
            test_trajectory,
        best_score=
            best_score,
        best_epoch=
            best_epoch,
    )

    with open(
        HISTORY_JSON,
        "w",
    ) as f:
        json.dump(
            history,
            f,
            indent=2,
        )

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_CSV,
        index=False,
    )

    pd.DataFrame(
        test_trajectory
    ).to_csv(
        TEST_TRAJECTORY_CSV,
        index=False,
    )


# ============================================================
# 17. SAVE FINAL TRAINING STATE
# ============================================================

last_epoch_completed = min(
    EPOCHS,
    max(
        [0]
        + [
            int(
                r[
                    "epoch"
                ]
            )
            for r in history
            if r.get(
                "epoch"
            ) is not None
        ]
    ),
)

save_checkpoint(
    FINAL_CKPT,
    epoch=
        last_epoch_completed,
    global_step=
        global_step,
    history=
        history,
    test_trajectory=
        test_trajectory,
    best_score=
        best_score,
    best_epoch=
        best_epoch,
)


# ============================================================
# 18. LOAD BEST TRAIN-MONITOR CHECKPOINT
# ============================================================

if not BEST_CKPT.exists():
    raise RuntimeError(
        "BEST checkpoint does not exist."
    )

best_ckpt = torch_load_full(
    BEST_CKPT
)

model.load_state_dict(
    best_ckpt[
        "model"
    ]
)

print("\n" + "=" * 120)
print("LOADED BEST TRAIN-MONITOR CHECKPOINT")
print("=" * 120)
print(
    "Best epoch :",
    best_ckpt[
        "best_epoch"
    ],
)
print(
    "Best train-monitor mean RelL2%:",
    best_ckpt[
        "best_score"
    ],
)


# ============================================================
# 19. FINAL FIXED 960-CASE TEST ON BEST CHECKPOINT
# ============================================================

best_fixed_test = eval_fixed_cases(
    fixed_test_cases,
    operator_level=True,
)

print("\nBEST checkpoint -- fixed 960-case unseen test:")

print(
    "  Pred RelL2%:",
    short_summary_str(
        best_fixed_test[
            "RelL2_summary_%"
        ]
    ),
)

print(
    "  Raw LF RelL2%:",
    short_summary_str(
        best_fixed_test[
            "RawUcRelL2_summary_%"
        ]
    ),
)

print(
    "  R2:",
    short_summary_str(
        best_fixed_test[
            "R2_summary"
        ]
    ),
)

print(
    "  Operator-mean RelL2%:",
    short_summary_str(
        best_fixed_test[
            "OperatorMeanRelL2_summary_%"
        ]
    ),
)


# ============================================================
# 20. INDEPENDENT FORMAL 1024-CASE HELDOUT TEST
# ============================================================

formal_result = eval_fixed_cases(
    formal_test_cases,
    operator_level=True,
    return_case_arrays=True,
)

formal_arrays = formal_result.pop(
    "_case_arrays"
)

formal_s = formal_result[
    "RelL2_summary_%"
]

formal_raw_s = formal_result[
    "RawUcRelL2_summary_%"
]

formal_r2_s = formal_result[
    "R2_summary"
]

formal_op_s = formal_result[
    "OperatorMeanRelL2_summary_%"
]

operator_rows = formal_result.pop(
    "operator_rows"
)

print("\n" + "=" * 120)
print("INDEPENDENT FORMAL HELDOUT TEST -- BEST CHECKPOINT")
print("=" * 120)

print(
    "  Pred RelL2%:",
    short_summary_str(
        formal_s
    ),
)

print(
    "  Raw LF RelL2%:",
    short_summary_str(
        formal_raw_s
    ),
)

print(
    "  R2:",
    short_summary_str(
        formal_r2_s
    ),
)

print(
    "  Operator-mean RelL2%:",
    short_summary_str(
        formal_op_s
    ),
)

formal_json_payload = {
    "run_tag":
        RUN_TAG,
    "dataset_path":
        str(
            DATASET_PATH
        ),
    "model_params":
        int(
            n_params
        ),
    "best_epoch":
        int(
            best_ckpt[
                "best_epoch"
            ]
        ),
    "best_train_monitor_mean_RelL2_pct":
        float(
            best_ckpt[
                "best_score"
            ]
        ),
    "n_formal_cases":
        FORMAL_TEST_CASES,
    "M":
        TEST_FIXED_DEMOS,
    "RelL2_summary_%":
        formal_s,
    "RawUcRelL2_summary_%":
        formal_raw_s,
    "R2_summary":
        formal_r2_s,
    "OperatorMeanRelL2_summary_%":
        formal_op_s,
}

with open(
    FORMAL_TEST_JSON,
    "w",
) as f:
    json.dump(
        formal_json_payload,
        f,
        indent=2,
    )

formal_case_df = pd.DataFrame({
    "case_id":
        np.arange(
            FORMAL_TEST_CASES,
            dtype=np.int64,
        ),
    "operator":
        formal_arrays[
            "op"
        ],
    "beta":
        beta_all[
            formal_arrays[
                "op"
            ]
        ],
    "trajectory":
        formal_arrays[
            "traj"
        ],
    "query_time_index":
        formal_arrays[
            "qtime_idx"
        ],
    "query_time":
        formal_arrays[
            "qtime"
        ],
    "pred_RelL2_pct":
        100.0
        * formal_arrays[
            "rel"
        ],
    "raw_LF_RelL2_pct":
        100.0
        * formal_arrays[
            "raw_rel"
        ],
    "R2":
        formal_arrays[
            "r2"
        ],
})

formal_case_df.to_csv(
    FORMAL_CASE_CSV,
    index=False,
)

pd.DataFrame(
    operator_rows
).to_csv(
    FORMAL_OPERATOR_CSV,
    index=False,
)


# ============================================================
# 21. TRAINING SUMMARY / PLOTS
# ============================================================

best_fixed_test_json = dict(
    best_fixed_test
)

best_fixed_test_json.pop(
    "operator_rows",
    None,
)

training_summary = {
    "run_tag":
        RUN_TAG,
    "dataset_path":
        str(
            DATASET_PATH
        ),
    "device":
        DEVICE,
    "model_params":
        int(
            n_params
        ),
    "model": {
        "d_model":
            D_MODEL,
        "n_head":
            N_HEAD,
        "n_layers":
            N_LAYERS,
        "d_ff":
            D_FF,
        "dropout":
            DROPOUT,
    },
    "training": {
        "batch_size":
            BATCH_SIZE,
        "epochs":
            EPOCHS,
        "steps_per_epoch":
            STEPS_PER_EPOCH,
        "weight_decay":
            WEIGHT_DECAY,
        "base_lr":
            BASE_LR,
        "max_lr":
            MAX_LR,
        "min_lr":
            MIN_LR,
        "warmup_frac":
            WARMUP_FRAC,
        "grad_clip":
            GRAD_CLIP,
    },
    "dataset": {
        "nx_lf":
            nx_lf_saved,
        "nx_hf":
            nx_hf_saved,
        "nr":
            nr_saved,
        "lf_solver_tag":
            lf_solver_tag_saved,
        "hf_solver_tag":
            hf_solver_tag_saved,
        "eligible_time_min":
            float(
                eligible_times[
                    0
                ]
            ),
        "eligible_time_max":
            float(
                eligible_times[
                    -1
                ]
            ),
        "n_eligible_times":
            N_ELIGIBLE_TIME,
    },
    "best_epoch":
        int(
            best_ckpt[
                "best_epoch"
            ]
        ),
    "best_train_monitor_mean_RelL2_pct":
        float(
            best_ckpt[
                "best_score"
            ]
        ),
    "best_fixed_test":
        best_fixed_test_json,
    "formal_test":
        formal_json_payload,
    "raw_full_split": {
        "train_RelL2_%":
            summarize_array(
                train_raw_rel,
                scale=100.0,
            ),
        "test_RelL2_%":
            summarize_array(
                test_raw_rel_full,
                scale=100.0,
            ),
    },
}

with open(
    FINAL_METRICS_JSON,
    "w",
) as f:
    json.dump(
        training_summary,
        f,
        indent=2,
    )


if len(
    test_trajectory
) > 0:
    traj_df = pd.DataFrame(
        test_trajectory
    )

    plt.figure(
        figsize=(
            7.5,
            5.0,
        )
    )

    plt.plot(
        traj_df[
            "epoch"
        ],
        traj_df[
            "mean_RelL2_pct"
        ],
        marker="o",
        label="heldout mean",
    )

    plt.plot(
        traj_df[
            "epoch"
        ],
        traj_df[
            "median_RelL2_pct"
        ],
        marker="s",
        label="heldout median",
    )

    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "RelL2 (%)"
    )

    plt.title(
        "KS FD2-LF72 -> spectral-HF256 fixed unseen-operator M=5 trajectory"
    )

    plt.legend()
    plt.tight_layout()

    plt.savefig(
        SAVE_DIR
        / f"{RUN_TAG}_fixed_test_trajectory.png",
        dpi=180,
    )

    plt.close()


# ============================================================
# 22. FINAL PRINT
# ============================================================

elapsed_minutes = (
    time.time()
    - training_start
) / 60.0

print("\n" + "=" * 120)
print("TRAINING COMPLETE")
print("=" * 120)
print("RUN_TAG      :", RUN_TAG)
print("DATASET      :", DATASET_PATH)
print("SAVE_DIR     :", SAVE_DIR)
print("MODEL PARAMS :", f"{n_params:,}")
print("BEST EPOCH   :", best_ckpt["best_epoch"])

print(
    "BEST TRAIN MONITOR MEAN RELL2:",
    f"{best_ckpt['best_score']:.4f}%",
)

print(
    "FORMAL TEST PRED RELL2:",
    short_summary_str(
        formal_s
    ),
)

print(
    "FORMAL TEST RAW LF RELL2:",
    short_summary_str(
        formal_raw_s
    ),
)

print(
    "FORMAL TEST R2:",
    short_summary_str(
        formal_r2_s
    ),
)

print(
    "FORMAL TEST OP-MEAN RELL2:",
    short_summary_str(
        formal_op_s
    ),
)

print(
    "TRAINING/RESUME WALL MINUTES THIS SESSION:",
    f"{elapsed_minutes:.2f}",
)

print("\nSaved:")

for p in [
    BEST_CKPT,
    LATEST_CKPT,
    FINAL_CKPT,
    HISTORY_JSON,
    HISTORY_CSV,
    TEST_TRAJECTORY_CSV,
    FINAL_METRICS_JSON,
    FORMAL_TEST_JSON,
    FORMAL_CASE_CSV,
    FORMAL_OPERATOR_CSV,
]:
    print(" ", p)

print("=" * 120)

KS QUERY-t / FD2-LF72 -> SPECTRAL-HF256 / WITH-t / MIXED-M PAD23 -- LARGE MODEL 288-H6-FF576
DEVICE       : cuda
DATASET_ROOT : /kaggle/input/notebooks/xinyuanli913/ks-eqn-dataset
SAVE_DIR     : /kaggle/working/ks_queryt_withT_beta070_085_T3p5_FD2LF72_SpecHF256_Nr100_large288_H6_L5_FF576_train140_test60_sameTraj_randomTime_mixedM_pad23_150ep_step900
TRAIN/TEST   : 140/60 beta operators from NPZ saved split
M train      : 1..5, independent M per episode
Episode      : same beta + same trajectory + M+1 distinct random eligible times
Model input  : x / t / uc / uf; t explicitly supplied as constant normalized row
Batching     : right-pad to 23 + src_key_padding_mask + true query gather
Train monitor: fixed M=5, 512 train-side cases; SELECTS best checkpoint
Fixed test   : M=5, 16/operator x 60 = 960 cases; diagnostic only
Formal test  : independent M=5, 1024 unseen-beta cases, BEST checkpoint only
Epochs       : 150

DATASET_PATH: /kaggle/input/notebooks/xinyuanli913/ks-eqn-dataset/ks_mf_q

/tmp/ipykernel_23/1469157331.py:1747: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(



Model:
  row width Nr    : 100
  d_model         : 288
  heads           : 6
  layers          : 5
  FF              : 576
  dropout         : 0.1
  trainable params: 3,484,036

Padding-information sanity check:
  M                 : 2
  valid length      : 11
  max |predA-predB| : 0.000e+00
  PASS: masked padded rows are not prediction information.

Optimizer/scheduler:
  epochs      : 150
  total steps : 135000
  warmup steps: 10800
  base LR     : 0.0001
  max LR      : 0.0002
  min LR      : 1e-05

Epoch 000 diagnostics:
  TRAIN monitor: min=97.593, max=108.517, mean=102.507, median=102.217, p90=105.146, std=2.064
  TEST heldout : min=97.527, max=108.835, mean=102.619, median=102.519, p90=105.285, std=2.042
  TEST raw LF  : min=0.935, max=213.723, mean=28.817, median=15.969, p90=70.860, std=35.076

Epoch 005 | loss=3.422561e-02 | lr=1.417e-04 | 258.0s
  TRAIN monitor RelL2%: min=2.644, max=65.743, mean=8.461, median=6.456, p90=14.456, std=6.293
  TEST heldout RelL2% : min=2.842, m